In [1]:
import mlflow
import pandas as pd

In [3]:
print("MLflow Version:", mlflow.__version__)
mlflow.set_tracking_uri("http://localhost:5000")
print("Tracking URI:", mlflow.tracking.get_tracking_uri())

MLflow Version: 3.1.4
Tracking URI: http://localhost:5000


In [2]:
df = pd.read_parquet("../data/2022-10-26_hiscore_data.parquet.gzip")
df.head()

,name,created_at,updated_at,possible_ban,confirmed_ban,label_id,label,account_status,id,timestamp,...,tombs_of_amascut,tombs_of_amascut_expert,tzkal_zuk,tztok_jad,venenatis,vetion,vorkath,wintertodt,zalcano,zulrah
Player_id,,,,,,,,,,,,,,,,,,,,,
1,3BA604236FB0319D5937E31388B0C64C,2021-03-14 20:22:45,2022-12-19 05:02:57,0,0,1,Real_Player,not banned,59568395,2022-12-19 05:04:16,...,0.0,0.0,0,0,0,0,0,0,0,0
8,5A02B5A7F38AD2623A9C5E68DF01EC2F,2021-03-14 20:42:37,2022-12-19 00:36:08,0,0,1,Real_Player,not banned,59622273,2022-12-19 00:36:09,...,0.0,0.0,0,10,0,0,114,73,0,1256
29,59DCFCAFC1F3DF3326F36E7A39B741FC,2021-03-14 22:17:16,2022-12-19 14:49:19,0,0,1,Real_Player,not banned,292513577,2022-12-19 14:49:20,...,0.0,0.0,0,0,0,0,0,0,0,0
39,1C74EFD6CE51790D7BF65A94F47675B5,2021-03-14 22:17:23,2022-12-19 17:46:42,0,0,1,Real_Player,not banned,59503415,2022-12-19 17:46:53,...,235.0,0.0,0,0,0,0,279,1526,145,0
59,E666957B20A95519E6306D75FEC4DE19,2021-03-14 22:17:40,2022-12-19 01:23:32,1,0,1,Real_Player,not banned,59615490,2022-07-27 06:41:21,...,0.0,0.0,0,20,0,0,1000,500,0,400


In [8]:
from sklearn.model_selection import train_test_split

def load_and_split_data(
    file_path,
    target_column,
    feature_columns,
    test_size=0.2,
    random_state=42,
):
    """
    Loads data from a parquet file, splits it into features (X) and target (y),
    and then into training and testing sets.

    Args:
        file_path (str): The path to the parquet data file.
        target_column (str): The name of the column to be used as the target variable.
        feature_columns (list): A list of column names to be used as features.
        test_size (float): The proportion of the dataset to allocate to the test split.
        random_state (int): The seed used by the random number generator.

    Returns:
        tuple: A tuple containing X_train, X_test, y_train, y_test.
    """
    print(f"Loading data from {file_path}...")
    df = pd.read_parquet(file_path)
    print(f"Data loaded with {len(df)} samples and {len(df.columns)} columns.")

    # Ensure all feature columns are present
    missing_features = [col for col in feature_columns if col not in df.columns]
    if missing_features:
        raise ValueError(f"Missing feature columns: {missing_features}")
    
    # Ensure target is boolean/integer (0 or 1)
    df[target_column] = df[target_column].astype(int)

    X = df[feature_columns]
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )
    print(f"Data split into {len(X_train)} training and {len(X_test)} testing samples.")

    return X_train, X_test, y_train, y_test, X, y

In [ ]:
DATA_FILE = "../data/2022-10-26_hiscore_data.parquet.gzip"
TARGET_COLUMN = "confirmed_ban"
FEATURE_COLUMNS = [
    "total",
    "attack",
    "defence",
    "strength",
    "hitpoints",
    "ranged",
    "prayer",
    "magic",
    "cooking",
    "woodcutting",
    "fletching",
    "fishing",
    "firemaking",
    "crafting",
    "smithing",
    "mining",
    "herblore",
    "agility",
    "thieving",
    "slayer",
    "farming",
    "runecraft",
    "hunter",
    "construction",
]

X_train, X_test, y_train, y_test, X, y = load_and_split_data(
    file_path=DATA_FILE,
    target_column=TARGET_COLUMN,
    feature_columns=FEATURE_COLUMNS,
)

Loading data from ../data/2022-10-26_hiscore_data.parquet.gzip...
Data loaded with 286234 samples and 97 columns.
Data split into 228987 training and 57247 testing samples.


In [16]:
X_test

,total,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,woodcutting,...,smithing,mining,herblore,agility,thieving,slayer,farming,runecraft,hunter,construction
Player_id,,,,,,,,,,,,,,,,,,,,,
49372229,241721106,19205589,16932566,38181126,39403029,28519539,5360019,19199032,16231954,3973859,...,2300544,2810204,3277765,2289352,2210624,5027805,6184978,1429524,1439817,2973933
20891521,10823537,287843,145850,302674,284422,65334,322572,3636739,111240,2650,...,112166,35,0,3015,1720,32438,0,0,0,7852
5682406,2164376,14009,101953,38493,52236,12,50537,9,70,24815,...,18,155477,0,147483,0,0,0,1564368,0,0
381143,63553568,2192839,1482315,5321590,4866317,4290804,818052,3143413,3261310,2349304,...,1135677,1622784,1214981,1416625,1334376,2160063,2039456,410225,3995742,1137833
5087225,5346217,37252,0,1986288,1332702,1989768,0,9,70,25,...,18,35,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548111,13002312,1210473,1210439,1991242,1771317,737724,550591,826806,300928,275425,...,820629,2241781,0,0,0,0,0,144970,0,0
4533443,61788385,5209180,3052081,5234109,8070448,11635705,739323,12775835,1164058,462478,...,1012594,5586596,528425,105279,1226435,432021,5110,11503,167167,9495
50071261,469409364,16527208,17995465,22698125,43500175,50987869,13479944,25912060,14399023,13104374,...,13410857,15040079,13900064,13043742,21528044,29097216,15760091,13654563,36852912,13200709


In [ ]:
from mlflow.models import infer_signature
import warnings

with warnings.catch_warnings():
    # Specifically ignore the UserWarning from MLflow about integer columns
    warnings.filterwarnings(
        "ignore"
    )
    infer_signature(
        model_input=X_train,
        model_output=y_train,
    )

2025/08/03 22:19:36 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
